# Phoenix Span Explorer

In [ ]:
# ── Load .env — run this cell first before any agent calls ───────────────
# Jupyter kernels do not inherit the shell .env automatically.
# This cell locates the repo-root .env relative to the notebook file
# and loads it explicitly so IBM_CLOUD_API_KEY and friends are set.
import os
from pathlib import Path
from dotenv import load_dotenv

# Walk up from this notebook to the repo root (.env sits two levels up from eval/phoenix/)
notebook_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else os.getcwd())
# Fallback: resolve relative to cwd and walk up until we find .env
search = Path(os.getcwd())
env_path = None
for _ in range(6):  # search up to 6 levels
    candidate = search / '.env'
    if candidate.exists():
        env_path = candidate
        break
    search = search.parent

if env_path:
    load_dotenv(env_path, override=True)
    print(f'Loaded .env from: {env_path}')
else:
    print('WARNING: .env not found — set IBM_CLOUD_API_KEY manually below')

# Verify required vars are present
required = ['IBM_CLOUD_API_KEY', 'ORCHESTRATE_BASE_URL', 'ORCHESTRATE_AGENT_ID']
missing = [v for v in required if not os.getenv(v)]
if missing:
    print(f'MISSING env vars: {missing}')
    print('Set them here if needed:')
    # os.environ['IBM_CLOUD_API_KEY'] = 'your-key-here'
else:
    print('All required env vars are set ✓')

In [11]:
from phoenix.client import Client

client = Client()
experiment = client.experiments.get(experiment_id = "RXhwZXJpbWVudDoxMA==")
experiment

{'id': 'RXhwZXJpbWVudDoxMA==',
 'dataset_id': 'RGF0YXNldDox',
 'dataset_version_id': 'RGF0YXNldFZlcnNpb246Mw==',
 'name': 'bls-eval-0-v1-run1',
 'description': None,
 'repetitions': 1,
 'metadata': {},
 'project_name': 'Experiment-4608f885007db06502f53919',
 'created_at': '2026-11-18T21:36:26+00:00',
 'updated_at': '2026-11-18T21:36:26+00:00',
 'example_count': 30,
 'successful_run_count': 30,
 'failed_run_count': 0,
 'missing_run_count': 0}

In [12]:
from phoenix.otel import register
tracer_provider = register(project_name = "Experiment-4608f885007db06502f53919", 
auto_instrument = True)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: Experiment-4608f885007db06502f53919
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [6]:
for project in client.projects.list():
    print(project['name'])
    name = project['name']
    spans_df= client.spans.get_spans_dataframe(project_identifier=name)
    if spans_df is not None and not spans_df.empty:
        print(spans_df['span_kind'].value_counts())

evaluators
span_kind
EVALUATOR    663
Name: count, dtype: int64
default


## Tool Usage Analysis

Uses `agent_v2.py` (SSE streaming endpoint) which captures tool call names, args, and responses per question.

**Cell 1** — quick single-question probe to verify tool call capture is working.  
**Cell 2** — sweep the full dataset and count tool call frequency.

In [ ]:
# ── Quick probe: ask one question and inspect tool calls ──────────────────
import sys
sys.path.insert(0, '.')  # ensure agent_v2 is importable when run from eval/phoenix/

from agent_v2 import OrchestrateAgentV2

agent = OrchestrateAgentV2()

question = 'What BLS series covers the national unemployment rate?'
result = await agent.run(question)

print(f'Question : {question}')
print(f'Answer   : {result.answer[:200]}')
print(f'Tool calls ({result.tool_call_count}):')
for tc in result.tool_calls:
    print(f'  name : {tc["name"]}')
    print(f'  args : {tc["args"]}')
    print(f'  id   : {tc["tool_call_id"]}')
print(f'Tool responses ({len(result.tool_responses)}):')
for tr in result.tool_responses:
    preview = str(tr['output'])[:120].replace('\n', ' ')
    print(f'  {tr["name"]}: {preview}...')

In [ ]:
# ── Full dataset sweep: tool call frequency across all 30 questions ───────
import csv
from collections import Counter
from pathlib import Path

from agent_v2 import OrchestrateAgentV2

DATASET_CSV = Path('datasets/bls-eval-0/examples/bls-eval-0.csv')

agent = OrchestrateAgentV2()
tool_freq: Counter = Counter()
questions_with_no_tools: list[str] = []

rows = list(csv.DictReader(DATASET_CSV.open()))
print(f'Running {len(rows)} questions through agent_v2 ...')

for idx, row in enumerate(rows, 1):
    q = row['query']
    r = await agent.run(q)
    names = [tc['name'] for tc in r.tool_calls]
    if names:
        tool_freq.update(names)
    else:
        questions_with_no_tools.append(q)
    print(f'  [{idx:02d}/{len(rows)}] {q[:60]:<60}  tools={names or "(none)"}')

print()
print('─' * 50)
print(f'Tool call frequency ({sum(tool_freq.values())} total calls across {len(rows)} questions):')
for name, count in tool_freq.most_common():
    pct = count / len(rows) * 100
    print(f'  {count:>4}x  ({pct:5.1f}%)  {name}')

if questions_with_no_tools:
    print(f'\nQuestions with NO tool calls ({len(questions_with_no_tools)}):')
    for q in questions_with_no_tools:
        print(f'  - {q}')